In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/biswasbarsha/model-building-on-a-synthetic-datasets/codetest_test.txt
/kaggle/input/datasets/biswasbarsha/model-building-on-a-synthetic-datasets/codetest_train.txt


The two synthetic datasets were generated using the same underlying data model. Your goal is to build a predictive model using the data in the training dataset to predict the withheld target values from the test set.


You may use any tools available to you for this task. Ultimately, we will assess predictive accuracy on the test set using the mean squared error metric. You should produce the following:

* A 1,000 x 1 text file containing 1 prediction per line for each record in the test dataset.
* A brief writeup describing the techniques you used to generate the predictions. Details such as important features and your estimates of predictive performance are helpful here, though not strictly necessary.
* (Optional) An implementable version of your model. What this would look like largely depends on the methods you used, but could include things like source code, a pickled Python object, a PMML file, etc. Please do not include any compiled executables.

## DATA LOADING & PREPARATION

In [2]:
df_train = pd.read_csv("/kaggle/input/datasets/biswasbarsha/model-building-on-a-synthetic-datasets/codetest_train.txt", sep='\t')
df_train

,target,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,...,f_244,f_245,f_246,f_247,f_248,f_249,f_250,f_251,f_252,f_253
0,3.066056,-0.653,0.255,-0.615,-1.833,-0.736,NaN,1.115,-0.171,-0.351,...,-1.607,-1.400,-0.920,-0.198,-0.945,-0.573,0.170,-0.418,-1.244,-0.503
1,-1.910473,1.179,-0.093,-0.556,0.811,-0.468,-0.005,-0.116,-1.243,1.985,...,1.282,0.032,-0.061,NaN,-0.061,-0.302,1.281,-0.850,0.821,-0.260
2,7.830711,0.181,-0.778,-0.919,0.113,0.887,-0.762,1.872,-1.709,0.135,...,-0.237,-0.660,1.073,-0.193,0.570,-0.267,1.435,1.332,-1.147,2.580
3,-2.180862,0.745,-0.245,-1.343,1.163,-0.169,-0.151,-1.100,0.225,1.223,...,0.709,-0.203,-0.136,-0.571,1.682,0.243,-0.381,0.613,1.033,0.400
4,5.462784,1.217,-1.324,-0.958,0.448,-2.873,-0.856,0.603,0.763,0.020,...,0.892,-0.433,-0.877,0.289,0.654,1.230,0.457,-0.754,-0.025,-0.931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,-1.371557,-0.297,-0.715,2.349,1.098,1.162,0.953,1.321,0.344,0.861,...,-0.259,1.222,0.622,-2.447,-0.156,2.028,-1.512,-0.967,1.136,-0.510
4996,-3.120233,0.391,0.581,1.532,2.415,1.563,-1.156,1.369,0.263,0.766,...,0.836,-2.046,-0.374,-1.768,0.785,-0.224,-0.157,0.267,0.872,-1.428
4997,0.013335,-0.802,0.720,0.345,-0.440,-1.054,0.092,-0.481,0.790,-0.028,...,-1.154,-0.476,-0.527,-1.431,-1.203,0.321,-0.351,0.099,0.774,1.045
4998,4.289620,0.195,-1.554,-0.203,0.965,-0.794,0.654,-1.008,-1.291,-2.035,...,-1.182,2.297,0.517,1.222,1.758,0.293,-0.820,0.032,-1.740,-1.816


In [3]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 255 entries, target to f_253
dtypes: float64(251), object(4)
memory usage: 9.7+ MB


There are 251 features which are of **Float** datatype and 4 features which are of **Object** datatype.

In [4]:
# Loop through the features of Object Datatype and print the unique values.
for col in df_train.select_dtypes(include=['object']):
    print(f"Feature: {col}")
    print(df_train[col].unique())
    print("-" * 20)

Feature: f_61
['b' 'a' 'd' 'c' nan 'e']
--------------------
Feature: f_121
['D' 'A' 'B' 'C' 'E' 'F' nan]
--------------------
Feature: f_215
['red' 'blue' 'orange' 'yellow' nan]
--------------------
Feature: f_237
['Canada' 'USA' 'Mexico' nan]
--------------------


In [5]:
# To check is there any null values in a dataset
df_train.isna().sum().sort_values(ascending=False)

f_71      132
f_201     131
f_157     124
f_15      122
f_211     121
         ... 
f_183      80
f_251      79
f_1        72
f_48       72
target      0
Length: 255, dtype: int64

There are null values in all the features. So, we'll replace the Null values of Float features to median of the feature and replace the object features to mode of the feature. Median is being used here instead of mean because it's less prone to outlier.

But before that, Let's split the dataset into df_train and y.

In [16]:
y = df_train['target']
df_train = df_train.drop(columns='target')

In [17]:
from sklearn.impute import SimpleImputer

numeric_col = df_train.select_dtypes(include=['float64']).columns
categorical_col = df_train.select_dtypes(include=['object']).columns

num_imputer = SimpleImputer(strategy='median')
df_train[numeric_col] = num_imputer.fit_transform(df_train[numeric_col])

cat_imputer = SimpleImputer(strategy='most_frequent')
df_train[categorical_col] = cat_imputer.fit_transform(df_train[categorical_col])

In [18]:
# Now check is there any null values left in a dataset
df_train.isna().sum().sort_values(ascending=False)

f_0      0
f_1      0
f_2      0
f_3      0
f_4      0
        ..
f_249    0
f_250    0
f_251    0
f_252    0
f_253    0
Length: 254, dtype: int64

Now the null value issue is now resolved. Now it's time to apply same thing in test dataset.

In [19]:
df_test = pd.read_csv("/kaggle/input/datasets/biswasbarsha/model-building-on-a-synthetic-datasets/codetest_test.txt", sep='\t')
df_test

,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,f_9,...,f_244,f_245,f_246,f_247,f_248,f_249,f_250,f_251,f_252,f_253
0,1.122,2.372,-1.303,-0.421,-1.724,0.206,0.449,1.305,-0.344,0.307,...,0.988,-0.472,0.522,-0.308,1.062,-0.839,0.819,0.342,-0.162,-1.123
1,0.645,-0.818,-1.193,0.286,0.946,2.001,-1.491,-0.162,-1.668,0.310,...,-0.020,0.764,-0.623,0.147,0.392,0.509,1.608,-0.321,-1.723,1.223
2,-0.661,0.373,1.160,0.448,NaN,-0.378,-0.645,0.713,1.900,0.627,...,0.156,-0.267,0.140,-0.908,0.093,0.227,-0.996,-0.665,0.444,-1.452
3,0.837,1.270,-1.421,-0.483,1.136,0.051,-0.097,NaN,-1.524,-0.253,...,1.653,1.951,0.657,-1.238,-0.487,-1.341,1.221,1.938,-0.447,0.411
4,-0.001,-0.050,0.442,0.230,0.525,0.326,-0.590,-0.287,-0.556,-0.088,...,-0.291,-0.196,-0.738,-0.377,-0.660,1.776,-1.028,-0.797,0.185,0.378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-1.190,0.260,-0.303,0.784,0.718,-0.683,0.070,-0.300,-1.627,-1.605,...,1.670,-0.618,1.584,-0.749,-0.410,-0.349,-0.065,0.469,-1.032,0.271
996,-1.215,0.740,0.769,0.266,0.088,0.730,1.065,0.409,0.316,1.494,...,1.271,1.029,1.115,-0.081,-0.237,0.113,-0.348,-0.373,-1.021,NaN
997,-0.610,0.019,-1.733,1.414,-1.148,-0.702,0.451,0.274,0.186,0.910,...,1.396,-0.517,2.018,0.306,0.800,0.504,0.764,1.592,-0.451,1.215
998,0.025,0.100,-0.077,0.343,NaN,0.360,1.688,-0.882,0.834,-0.595,...,-0.089,0.564,2.365,0.529,NaN,0.194,NaN,-0.512,-2.358,-0.413


In [20]:
# To check is there any null values in a dataset
df_test.isna().sum().sort_values(ascending=False)

f_144    34
f_14     32
f_95     31
f_117    30
f_73     29
         ..
f_59     11
f_217    11
f_85     10
f_122    10
f_186    10
Length: 254, dtype: int64

In [21]:
df_test[numeric_col] = num_imputer.transform(df_test[numeric_col])
df_test[categorical_col] = cat_imputer.transform(df_test[categorical_col])

In [22]:
# Now check is there any null values left in a dataset
df_test.isna().sum().sort_values(ascending=False)

f_0      0
f_1      0
f_2      0
f_3      0
f_4      0
        ..
f_249    0
f_250    0
f_251    0
f_252    0
f_253    0
Length: 254, dtype: int64

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(df_train, y, test_size=0.2, random_state=42)

In [31]:
# For training dataset:

# 'ignore' ensures that if the validation set has a weird, unseen category, the model won't crash!
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# (Assuming categorical_col holds your categorical column names)
encoded_train = encoder.fit_transform(X_train[categorical_col])
encoded_df_train = pd.DataFrame(encoded_train, columns=encoder.get_feature_names_out(categorical_col), index=X_train.index)

X_train = X_train.drop(columns=categorical_col)
X_train = pd.concat([X_train, encoded_df_train], axis=1)

In [35]:
# For val dataset: 

# 1. Transform the validation categorical columns (using the encoder you ALREADY fitted)
encoded_val = encoder.transform(X_val[categorical_col])

# 2. Convert it into a clean DataFrame with the correct column names
encoded_df_val = pd.DataFrame(
    encoded_val, 
    columns=encoder.get_feature_names_out(categorical_col), 
    index=X_val.index
)

# 3. Drop the old text columns from X_val
X_val = X_val.drop(columns=categorical_col)

# 4. Attach the new encoded columns
X_val = pd.concat([X_val, encoded_df_val], axis=1)

In [33]:
# For test dataset:

# 1. Encode the test data using the existing encoder (NO fitting)
encoded_test = encoder.transform(df_test[categorical_col])
encoded_df_test = pd.DataFrame(
    encoded_test, 
    columns=encoder.get_feature_names_out(categorical_col), 
    index=df_test.index
)

# 2. Drop the old text columns and combine
df_test = df_test.drop(columns=categorical_col)
df_test = pd.concat([df_test, encoded_df_test], axis=1)

## MODEL TRAINING

### 1.Linear Regression(Multivariable Regression)

In [32]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

### 2.Lasso Regression

In [40]:
from sklearn.linear_model import Lasso

lasso = Lasso()
lasso.fit(X_train, y_train)

Lasso()

### 2.Ridge Regression

In [41]:
from sklearn.linear_model import Ridge

ridge = Ridge()
ridge.fit(X_train, y_train)

Ridge()

## Model Evaluation

In [46]:
# Calculate the scores for this model
print("*************Linear Regression**************")

y_pred = lr.predict(X_val)
# R^2 Score
from sklearn.metrics import r2_score
lr_r2 = r2_score(y_val, y_pred)
print("R2 Score: ", lr_r2)

# Adjusted R^2 Score
n = X_val.shape[0]
p = X_test.shape[1]

adj_r2_lr = 1-((1-lr_r2)*(n-1))/(n-p-1)
print("Adjusted R2 Score: ", adj_r2_lr)

# Mean Squared Error
from sklearn.metrics import mean_squared_error
mse_lr = mean_squared_error(y_test, y_pred)
print("Mean Square Error: ", mse_lr)

# Root Mean Squared Error
from sklearn.metrics import root_mean_squared_error
rmse_lr = root_mean_squared_error(y_test, y_pred)
print("Root Mean Square Error: ", rmse_lr)

# Mean Absolute Error
from sklearn.metrics import mean_absolute_error
mae_lr = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error: ", mae_lr)

# Mean Absolute Percentage Error
from sklearn.metrics import mean_absolute_percentage_error
mape_lr = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error: ", mape_lr)

# Median Absolute Error
from sklearn.metrics import median_absolute_error
medae_lr = median_absolute_error(y_test, y_pred)
print("Median Absolute Error: ", medae_lr)
print("\n")

print("*************Lasso Regression**************")

y_pred = lr.predict(X_val)
# R^2 Score
from sklearn.metrics import r2_score
la_r2 = r2_score(y_val, y_pred)
print("R2 Score: ", la_r2)

# Adjusted R^2 Score
n = X_val.shape[0]
p = X_test.shape[1]

adj_r2_la = 1-((1-lr_r2)*(n-1))/(n-p-1)
print("Adjusted R2 Score: ", adj_r2_la)

# Mean Squared Error
from sklearn.metrics import mean_squared_error
mse_la = mean_squared_error(y_test, y_pred)
print("Mean Square Error: ", mse_la)

# Root Mean Squared Error
from sklearn.metrics import root_mean_squared_error
rmse_la = root_mean_squared_error(y_test, y_pred)
print("Root Mean Square Error: ", rmse_la)

# Mean Absolute Error
from sklearn.metrics import mean_absolute_error
mae_la = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error: ", mae_la)

# Mean Absolute Percentage Error
from sklearn.metrics import mean_absolute_percentage_error
mape_la = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error: ", mape_la)

# Median Absolute Error
from sklearn.metrics import median_absolute_error
medae_la = median_absolute_error(y_test, y_pred)
print("Median Absolute Error: ", medae_la)
print("\n")

print("*************Ridge Regression**************")

y_pred = lr.predict(X_val)
# R^2 Score
from sklearn.metrics import r2_score
r_r2 = r2_score(y_val, y_pred)
print("R2 Score: ", r_r2)

# Adjusted R^2 Score
n = X_val.shape[0]
p = X_test.shape[1]

adj_r2_r = 1-((1-lr_r2)*(n-1))/(n-p-1)
print("Adjusted R2 Score: ", adj_r2_r)

# Mean Squared Error
from sklearn.metrics import mean_squared_error
mse_r = mean_squared_error(y_test, y_pred)
print("Mean Square Error: ", mse_r)

# Root Mean Squared Error
from sklearn.metrics import root_mean_squared_error
rmse_r = root_mean_squared_error(y_test, y_pred)
print("Root Mean Square Error: ", rmse_r)

# Mean Absolute Error
from sklearn.metrics import mean_absolute_error
mae_r = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error: ", mae_r)

# Mean Absolute Percentage Error
from sklearn.metrics import mean_absolute_percentage_error
mape_r = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error: ", mape_r)

# Median Absolute Error
from sklearn.metrics import median_absolute_error
medae_r = median_absolute_error(y_test, y_pred)
print("Median Absolute Error: ", medae_r)


*************Linear Regression**************
R2 Score:  0.55230974837617
Adjusted R2 Score:  0.3996744142654951
Mean Square Error:  12.682596855473726
Root Mean Square Error:  3.5612633791217587
Mean Absolute Error:  2.5320864771344382
Mean Absolute Percentage Error:  3.6278533766662653
Median Absolute Error:  1.8013283009613539


*************Lasso Regression**************
R2 Score:  0.55230974837617
Adjusted R2 Score:  0.3996744142654951
Mean Square Error:  12.682596855473726
Root Mean Square Error:  3.5612633791217587
Mean Absolute Error:  2.5320864771344382
Mean Absolute Percentage Error:  3.6278533766662653
Median Absolute Error:  1.8013283009613539


*************Ridge Regression**************
R2 Score:  0.55230974837617
Adjusted R2 Score:  0.3996744142654951
Mean Square Error:  12.682596855473726
Root Mean Square Error:  3.5612633791217587
Mean Absolute Error:  2.5320864771344382
Mean Absolute Percentage Error:  3.6278533766662653
Median Absolute Error:  1.8013283009613539


## Final Evaluation of Baseline Linear Models
1. **Baseline Predictive Power:**The baseline linear models (Standard, Lasso, and Ridge) consistently explain approximately 55.2% of the variance ($R^2 = 0.552$) in the target variable. The models produce an average prediction error of roughly 3.56 units (RMSE), with an absolute typical error margin of 2.53 units (MAE).
2. **The Regularization Impact (or lack thereof):** The fact that Linear Regression, Lasso (L1), and Ridge (L2) yielded identically matched scores down to the decimal is a critical insight. This indicates that applying standard linear regularization provided zero performance gains. This typically happens when the dataset lacks severe multicollinearity, or when the default penalty parameters ($\alpha$) are too small to meaningfully shrink the feature coefficients.
3. **Conclusion on the Data Structure:** An $R^2$ of 0.55 is a respectable baseline, but it clearly indicates that a purely linear approach leaves 45% of the variance unexplained. This strongly suggests that the underlying synthetic data model contains non-linear relationships and complex feature interactions. While these linear equations provide a solid, stable benchmark (an MSE of 12.68), capturing the remaining variance and driving the error down further would fundamentally require transitioning to a non-linear algorithm.